In [0]:
import uuid
import time
import json
from datetime import datetime
from pyspark.sql.functions import current_timestamp, col, lit

In [0]:
dbutils.widgets.text("CATALOG","")
dbutils.widgets.text("VOLUME_PATH","")
dbutils.widgets.text("CHECKPOINT_BASE","")
dbutils.widgets.text("AUDIT_TABLE","")


In [0]:
CATALOG = dbutils.widgets.get("CATALOG")
VOLUME_PATH = dbutils.widgets.get("VOLUME_PATH")
CHECKPOINT_BASE = dbutils.widgets.get("CHECKPOINT_BASE")
AUDIT_TABLE = dbutils.widgets.get("AUDIT_TABLE")

In [0]:
def write_audit_row(row):
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType
    
    # Define explicit schema matching the audit table
    audit_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("layer", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("source_path", StringType(), True),
        StructField("status", StringType(), True),
        StructField("rows_ingested", LongType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_seconds", DoubleType(), True),
        StructField("error_message", StringType(), True),
        StructField("triggered_by", StringType(), True),
        StructField("metadata", StringType(), True)
    ])
    
    # Convert metadata dict to JSON string
    row_copy = row.copy()
    if 'metadata' in row_copy and isinstance(row_copy['metadata'], dict):
        row_copy['metadata'] = json.dumps(row_copy['metadata'])
    
    spark.createDataFrame([row_copy], schema=audit_schema).write.format('delta').mode('append').saveAsTable(AUDIT_TABLE)

In [0]:
def ingest_bronze(source_file, table_name, run_id, file_format='csv'):
    source_path = f"{VOLUME_PATH}/{source_file}/"
    schema_loc = f"{CHECKPOINT_BASE}/{table_name}/_schema"
    checkpoint_loc = f"{CHECKPOINT_BASE}/{table_name}/_checkpoint"
    target_table = f"{CATALOG}.bronze.{table_name}"

    start = datetime.utcnow()
    start_ts = time.time()

    audit_row = {
        "run_id": run_id,
        "layer": "bronze",
        "table_name": table_name,
        "source_path": source_path,
        "status": "STARTED",
        "rows_ingested": None,
        "start_time": start,
        "end_time": None,
        "duration_seconds": None,
        "error_message": None,
        "triggered_by": spark.sql("SELECT current_user()").collect()[0][0],
        "metadata":{
            "file_format": file_format,
            "schema_evolution_mode": "addNewColumns",
            "target_table": target_table,
        },
    }

    try:
        reader = (
            spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", file_format)
                .option("cloudFiles.schemaLocation", schema_loc)
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaEvolutionMode","addNewColumns")
                .option("pathGlobFilter", source_file)
                  .option("header", "true")
                  .option("cloudFiles.inferSchema", "true")
        )

        if file_format == 'csv':
            reader = reader.option("header","true").option("cloudFiles.inferSchema","true")

        df = (
                reader.load(VOLUME_PATH)
                .withColumn("_ingested_at", current_timestamp())
                .withColumn("_source_file", col("_metadata.file_path"))
                .withColumn("_source_table", lit(table_name))
              )
        query = (
            df.writeStream
                .format("delta")
                .option("checkpointLocation", checkpoint_loc)
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .toTable(target_table)
        )

        query.awaitTermination()

        rows_written = sum(
        p.get("numOutputRows", 0)
        for b in query.recentProgress
        for p in [b.get("sink", {})] if p
        )

        if rows_written is None or rows_written < 0:
            rows_written = spark.table(target_table).count()

        files_processed = len({b.get("sourcePath") for b in query.recentProgress if b.get("sourcePath")})

        end = datetime.utcnow()
        audit_row.update({
            "status":"SUCCESS",
            "rows_ingested": rows_written,
            "end_time": end,
            "duration_seconds": time.time() - start_ts,
        })

        audit_row['metadata']['files_processed'] = str(files_processed)
        audit_row['metadata']['batch_id'] = str(query.recentProgress[-1].get("batchId")) if query.recentProgress else None

    except Exception as e:
        end = datetime.utcnow()
        audit_row.update({
            "status": "FAILED",
            "end_time": end,
            "duration_seconds": time.time() - start_ts,
            "error_message": str(e)
        })

    write_audit_row(audit_row)
    return audit_row

In [0]:
bronze_tables = [
    {"source_file": "olist_customers.csv",              "table": "customers"},
    {"source_file": "olist_geolocation.csv",         "table": "geolocation"},
    {"source_file": "olist_order_items.csv",      "table": "order_items"},
    {"source_file": "olist_order_payments.csv",       "table": "order_payments"},
    {"source_file": "olist_order_reviews.csv",           "table": "order_reviews"},
    {"source_file": "olist_orders.csv",            "table": "orders"},
    {"source_file": "olist_products.csv",             "table": "products"},
    {"source_file": "olist_sellers.csv",         "table": "sellers"},
    {"source_file": "product_category_name_translation.csv", "table": "category_translation"},
]

In [0]:
run_id = str(uuid.uuid4())
print(f"Bronze ingestion run: {run_id}")

summary = []
for cfg in bronze_tables:
    print(f"Ingesting: {cfg['table']}...")
    result = ingest_bronze(cfg['source_file'], cfg['table'], run_id=run_id)
    summary.append(result)
    print(f"====== {result['status']} | rows: {result['rows_ingested']} | {result['duration_seconds']:.1f}s ======")

print('\nRun compted.')
display(spark.table("ecommerce_dev.audit.ingestion_log").filter(f"run_id = '{run_id}'"))

Bronze ingestion run: 5243dd0c-b6c3-4f38-8acf-1052a78238c6
Ingesting: customers...


/home/spark-0cf6b77a-0c49-493b-9229-dc/.ipykernel/81/command-8982522712238175-3970114272:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start = datetime.utcnow()
/home/spark-0cf6b77a-0c49-493b-9229-dc/.ipykernel/81/command-8982522712238175-3970114272:73: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end = datetime.utcnow()


====== SUCCESS | rows: 99441 | 4.3s ======
Ingesting: geolocation...
====== SUCCESS | rows: 1000163 | 4.1s ======
Ingesting: order_items...
====== SUCCESS | rows: 112650 | 4.1s ======
Ingesting: order_payments...
====== SUCCESS | rows: 103886 | 4.4s ======
Ingesting: order_reviews...
====== SUCCESS | rows: 104162 | 4.1s ======
Ingesting: orders...
====== SUCCESS | rows: 99441 | 4.2s ======
Ingesting: products...
====== SUCCESS | rows: 32951 | 4.1s ======
Ingesting: sellers...
====== SUCCESS | rows: 3095 | 4.3s ======
Ingesting: category_translation...
====== SUCCESS | rows: 71 | 4.0s ======

Run compted.


run_id,layer,table_name,source_path,status,rows_ingested,start_time,end_time,duration_seconds,error_message,triggered_by,metadata
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,category_translation,/Volumes/ecommerce_dev/raw_data/landing/product_category_name_translation.csv/,SUCCESS,71,2026-08-04T01:04:55.212Z,2026-08-04T01:04:59.178Z,3.9664723873138428,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.category_translation"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,order_payments,/Volumes/ecommerce_dev/raw_data/landing/olist_order_payments.csv/,SUCCESS,103886,2026-08-04T01:04:27.214Z,2026-08-04T01:04:31.603Z,4.389347553253174,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.order_payments"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,order_reviews,/Volumes/ecommerce_dev/raw_data/landing/olist_order_reviews.csv/,SUCCESS,104162,2026-08-04T01:04:33.090Z,2026-08-04T01:04:37.156Z,4.06606912612915,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.order_reviews"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,order_items,/Volumes/ecommerce_dev/raw_data/landing/olist_order_items.csv/,SUCCESS,112650,2026-08-04T01:04:21.645Z,2026-08-04T01:04:25.772Z,4.127185583114624,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.order_items"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,geolocation,/Volumes/ecommerce_dev/raw_data/landing/olist_geolocation.csv/,SUCCESS,1000163,2026-08-04T01:04:16.205Z,2026-08-04T01:04:20.270Z,4.0647053718566895,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.geolocation"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,customers,/Volumes/ecommerce_dev/raw_data/landing/olist_customers.csv/,SUCCESS,99441,2026-08-04T01:04:10.500Z,2026-08-04T01:04:14.814Z,4.314682960510254,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.customers"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,products,/Volumes/ecommerce_dev/raw_data/landing/olist_products.csv/,SUCCESS,32951,2026-08-04T01:04:44.096Z,2026-08-04T01:04:48.151Z,4.055415391921997,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.products"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,sellers,/Volumes/ecommerce_dev/raw_data/landing/olist_sellers.csv/,SUCCESS,3095,2026-08-04T01:04:49.573Z,2026-08-04T01:04:53.915Z,4.34245491027832,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.sellers"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
5243dd0c-b6c3-4f38-8acf-1052a78238c6,bronze,orders,/Volumes/ecommerce_dev/raw_data/landing/olist_orders.csv/,SUCCESS,99441,2026-08-04T01:04:38.446Z,2026-08-04T01:04:42.637Z,4.19097113609314,null,riju9826@gmail.com,"{""file_format"": ""csv"", ""schema_evolution_mode"": ""addNewColumns"", ""target_table"": ""ecommerce_dev.bronze.orders"", ""files_processed"": ""0"", ""batch_id"": ""1""}"
